# Notebook 4: Real-Time PL-Accelerated FFT & Spectrum Analyzer

This notebook demonstrates how to capture and analyze frequency spectra computed entirely within the **Programmable Logic (PL)** using the **Xilinx LogiCORE FFT (2048-pt)** and **CORDIC Magnitude Engine** on the PYNQ-Z2.

## 1. System Setup & Permission Check

In [ ]:
from pynq_oscilloscope import check_usb_permissions, OscilloscopeOverlay

# Ensure USB permissions are granted for AD3
check_usb_permissions()

## 2. Load the Hardware Overlay (v1.2.0-rc1)
Instantiate `OscilloscopeOverlay()`. It automatically fetches the `v1.2.0-rc1` bitstream featuring the dual-DMA architecture (Time DMA + FFT DMA).

In [ ]:
ol = OscilloscopeOverlay()
print("Oscilloscope & Spectrum Analyzer Overlay loaded successfully!")
print(f"Time-Domain Packet Size: {ol.packet_size} samples")
print(f"Frequency-Domain FFT Points: {ol.fft_points} points (1024 bins, 0 to 500 kHz)")

## 3. Generate a Test Tone with Analog Discovery 3
Generate a **25 kHz Sine wave** (1.5V Amplitude, 1.65V DC Offset) on Wavegen 1 (W1 $\rightarrow$ A0).

In [ ]:
import time

# Start 25 kHz Sine wave
ol.wavegen.start(shape="Sine", frequency=25000.0, amplitude=1.5, offset=1.65)
time.sleep(1.0)
print("AD3 signal generator active @ 25 kHz.")

## 4. Capture PL Hardware FFT Spectrum
Capture the spectrum directly from `axi_dma_1` in **dBV** units.

In [ ]:
from pynq_oscilloscope.fft_dma import StreamingFFT

# Capture single-sided spectrum (0 Hz to 500 kHz)
freqs, mags = ol.capture_fft(unit="dBV")

# Detect peak frequency
peak_freq, peak_mag = StreamingFFT.get_peak_frequency(freqs, mags, min_freq_hz=1000.0)
print(f"Dominant Peak: {peak_freq/1e3:.2f} kHz @ {peak_mag:.1f} dBV")

## 5. Spectrum Visualization
Plot the hardware spectrum using Matplotlib.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4.5), dpi=100)
ax.plot(freqs / 1e3, mags, color="#FF007F", linewidth=1.5, label="PL FFT Spectrum (2048-pt)")
ax.scatter([peak_freq / 1e3], [peak_mag], color="#00FFCC", s=60, zorder=5, label=f"Peak: {peak_freq/1e3:.1f} kHz ({peak_mag:.1f} dBV)")

ax.set_title("Hardware-Accelerated 1 MSPS FFT Spectrum (PL)", fontsize=12, fontweight="bold")
ax.set_xlabel("Frequency (kHz)", fontsize=10)
ax.set_ylabel("Magnitude (dBV)", fontsize=10)
ax.set_xlim(0, 100)  # Zoom to 0 - 100 kHz
ax.set_ylim(-70, 15)
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

## 6. Harmonic Distortion Test (Square Wave Harmonics)
Update AD3 to a **10 kHz Square wave** and observe the fundamental frequency ($f_0$) along with odd harmonics ($3f_0 = 30\,\text{kHz}$, $5f_0 = 50\,\text{kHz}$, $7f_0 = 70\,\text{kHz}$).

In [ ]:
# Switch to 10 kHz Square wave
ol.wavegen.update_parameters(shape="Square", frequency=10000.0, amplitude=1.2)
time.sleep(0.5)

freqs_sq, mags_sq = ol.capture_fft(unit="dBV")

fig, ax = plt.subplots(figsize=(10, 4.5), dpi=100)
ax.plot(freqs_sq / 1e3, mags_sq, color="#E040FB", linewidth=1.5, label="10 kHz Square Wave Spectrum")

ax.set_title("Square Wave Odd Harmonics (f0=10k, 3f0=30k, 5f0=50k, 7f0=70k)", fontsize=12, fontweight="bold")
ax.set_xlabel("Frequency (kHz)", fontsize=10)
ax.set_ylabel("Magnitude (dBV)", fontsize=10)
ax.set_xlim(0, 100)
ax.set_ylim(-70, 15)
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

## 7. Clean Hardware Shutdown

In [ ]:
ol.close()
print("Hardware closed cleanly.")